# Phase 2: Data Preprocessing Pipeline
This notebook demonstrates the memory-efficient preprocessing pipeline designed for the Satellite Image Land-Use Classifier. Phase 2 (Preprocessing) is designed to run entirely locally in the Antigravity IDE, while remaining compatible with Google Colab for Phase 3 (Training).This notebook demonstrates the memory-efficient preprocessing pipeline designed for the Satellite Image Land-Use Classifier. It ensures compatibility with Google Colab Free memory constraints.

In [ ]:
import os
import sys
import gc
import psutil
import numpy as np
import matplotlib.pyplot as plt
import torch

# Add root project to python path
sys.path.append(os.path.abspath('..'))
from src.preprocessing.preprocess import load_bands, extract_patches
from training.dataset import SatelliteDataset, get_data_loaders

## 1. Memory Monitor
We use `psutil` to track RAM usage through each step to verify memory efficiency.

In [ ]:
def print_memory_usage(step_name=""):
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    ram_usage = mem_info.rss / (1024 ** 2) # in MB
    print(f"[{step_name}] Memory Usage: {ram_usage:.2f} MB")

print_memory_usage("Initial Start")

## 2. Generate Dummy Data for Demonstration
If you have actual Sentinel-2 data, replace the `dummy_bands` paths with your files.

In [ ]:
import rasterio
from rasterio.transform import from_origin

os.makedirs("dummy_data", exist_ok=True)
dummy_bands = {'B2': 'dummy_data/B2.tif', 'B3': 'dummy_data/B3.tif', 'B4': 'dummy_data/B4.tif', 'B8': 'dummy_data/B8.tif'}

# Generate dummy 500x500 images (simulate a small satellite tile)
width, height = 500, 500
transform = from_origin(0, 0, 10, 10)
for band_name, path in dummy_bands.items():
    if not os.path.exists(path):
        data = np.random.randint(100, 8000, (height, width), dtype=np.uint16)
        with rasterio.open(path, 'w', driver='GTiff', height=height, width=width, count=1, 
                           dtype=data.dtype, crs='+proj=latlong', transform=transform) as dst:
            dst.write(data, 1)

print_memory_usage("After creating dummy data")

## 3. Patch Extraction, Normalization, NDVI & NDWI
The generator reads chunks of the image lazily using `rasterio.windows.Window`.

In [ ]:
# Initialize generator
patch_generator = extract_patches(dummy_bands, patch_size=64, stride=64)
print_memory_usage("Generator Initialized")

for i, patch_dict in enumerate(patch_generator):
    patch = patch_dict['image_patch']
    coords = patch_dict['coords']
    
    if i == 0:
        print(f"\nPatch 0 shape: {patch.shape}, DataType: {patch.dtype}, Coords: {coords}")
        print("Channels: Blue, Green, Red, NIR, NDVI, NDWI")
        
        # Visualize RGB (B4, B3, B2 are indices 2, 1, 0)
        rgb = patch[:, :, [2, 1, 0]]
        rgb_display = np.clip(rgb / (np.max(rgb) + 1e-6), 0, 1) # Simple stretching for display
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(rgb_display)
        axes[0].set_title("RGB Composite")
        
        # NDVI (index 4)
        ndvi = patch[:, :, 4]
        im1 = axes[1].imshow(ndvi, cmap='RdYlGn', vmin=-1, vmax=1)
        axes[1].set_title("NDVI")
        plt.colorbar(im1, ax=axes[1])
        
        # NDWI (index 5)
        ndwi = patch[:, :, 5]
        im2 = axes[2].imshow(ndwi, cmap='Blues', vmin=-1, vmax=1)
        axes[2].set_title("NDWI")
        plt.colorbar(im2, ax=axes[2])
        
        plt.show()
        
    if i > 5: # Process just a few patches to demonstrate generator state
        break

print_memory_usage("After partial patch extraction loop")

## 4. PyTorch Dataset & DataLoader
Demonstrating the `SatelliteDataset` with automatic 70/15/15 splits, data augmentation, and lazy loading.

In [ ]:
# Create some dummy saved patches to act as an ImageFolder dataset
os.makedirs("dummy_dataset/Forest", exist_ok=True)
os.makedirs("dummy_dataset/Water", exist_ok=True)
os.makedirs("dummy_dataset/Urban", exist_ok=True)

for i in range(10):
    np.save(f"dummy_dataset/Forest/patch_{i}.npy", np.random.rand(64, 64, 6).astype(np.float32))
    np.save(f"dummy_dataset/Water/patch_{i}.npy", np.random.rand(64, 64, 6).astype(np.float32))
    np.save(f"dummy_dataset/Urban/patch_{i}.npy", np.random.rand(64, 64, 6).astype(np.float32))

dataset = SatelliteDataset(root_dir="dummy_dataset")
print(f"Total dataset size: {len(dataset)} samples")
print(f"Discovered classes: {dataset.classes}")

# Load one item directly
image, label = dataset[0]
print(f"\nSample 0 -> Image tensor shape: {image.shape}, Label: {label}")

# Create dataloaders
train_loader, val_loader, test_loader = get_data_loaders("dummy_dataset", batch_size=4, num_workers=0)
print(f"\nTrain batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

# Iterate DataLoader
for batch_imgs, batch_labels in train_loader:
    print(f"Train Batch shape: {batch_imgs.shape} - Data type: {batch_imgs.dtype}")
    break

print_memory_usage("Final Dataset Demonstration")
gc.collect()